## GNN Project - SAGE
This notebook builds and evaluates a GraphSAGE model on the Amazon-ratings graph dataset.
It covers data loading, heterophily analysis, model training, hyperparameter tuning, and topology-based error analysis.

In [ ]:
# Install required packages.
import os
import torch
os.environ['TORCH'] = torch.__version__
print(torch.__version__)

!pip install -q torch-scatter -f https://data.pyg.org/whl/torch-${TORCH}.html
!pip install -q torch-sparse -f https://data.pyg.org/whl/torch-${TORCH}.html
!pip install -q git+https://github.com/pyg-team/pytorch_geometric.git

# Helper function for visualization.
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

def visualize(h, color, max_nodes=3000):
    # Move tensors to CPU and convert to numpy
    h_np = h.detach().cpu().numpy()
    color_np = color.cpu().numpy()

    # Subsample if the dataset is too large
    if h_np.shape[0] > max_nodes:
        print(f"Subsampling {max_nodes} nodes for faster visualization...")
        # Randomly select 'max_nodes' indices
        indices = np.random.choice(h_np.shape[0], max_nodes, replace=False)
        h_np = h_np[indices]
        color_np = color_np[indices]

    # Run t-SNE on the smaller subset
    z = TSNE(n_components=2, init='pca', learning_rate='auto').fit_transform(h_np)

    plt.figure(figsize=(10,10))
    plt.xticks([])
    plt.yticks([])

    plt.scatter(z[:, 0], z[:, 1], s=70, c=color_np, cmap="Set2", alpha=0.8)
    plt.show()


## Dataset Loading and Basic Inspection
In this section, we load the Amazon-ratings dataset from PyTorch Geometric and print core dataset statistics.
These checks confirm the number of nodes, edges, features, classes, and split structure before modeling.

In [ ]:
from torch_geometric.datasets import HeterophilousGraphDataset
from torch_geometric.transforms import NormalizeFeatures

dataset = HeterophilousGraphDataset(root='data/Amazon-ratings', name='Amazon-ratings', transform=NormalizeFeatures())

print()
print(f'Dataset: {dataset}:')
print('======================')
print(f'Number of graphs: {len(dataset)}')
print(f'Number of features: {dataset.num_features}')
print(f'Number of classes: {dataset.num_classes}')

data = dataset[0]  # Get the first graph object.

print()
print(data)
print('===========================================================================================================')

# Gather some statistics about the graph.
print(f'Number of nodes: {data.num_nodes}')
print(f'Number of edges: {data.num_edges}')
print(f'Average node degree: {data.num_edges / data.num_nodes:.2f}')
single_split_mask = data.train_mask[:,0]
print(f'Number of training nodes in one train split: {single_split_mask.sum()}')
print(f'Training node label rate: {int(single_split_mask.sum()) / data.num_nodes:.2f}')
print(f'Has isolated nodes: {data.has_isolated_nodes()}')
print(f'Has self-loops: {data.has_self_loops()}')
print(f'Is undirected: {data.is_undirected()}')

## Heterophily Analysis
This section measures how strongly the graph is heterophilic, meaning connected nodes often belong to different classes.
Understanding this helps explain why the classification task is challenging for standard SAGE behavior.

In [ ]:
# 1. DATASET PROFILING
import torch
from torch_geometric.utils import homophily

# Assuming your loaded dataset object is named 'data'
# The Amazon-ratings dataset is highly heterophilic. Let's quantify it:
edge_homo = homophily(data.edge_index, data.y, method='edge')
node_homo = homophily(data.edge_index, data.y, method='node')

print(f"Dataset Edge Homophily Ratio: {edge_homo:.4f}")
print(f"Dataset Node Homophily Ratio: {node_homo:.4f}")
print("Note: Values close to 0 indicate strong heterophily (nodes connect to different classes).")

## Training a SAGE
Here we prepare graph tensors, define the SAGE architecture, and run split-based training and evaluation.
We report both Accuracy and Macro-F1 to capture overall correctness and class-balanced performance.

In [ ]:
# Re-load dataset without NormalizeFeatures to stay closer to the original repo setup.
import torch
import numpy as np
from torch_geometric.datasets import HeterophilousGraphDataset
from torch_geometric.utils import to_undirected, degree

torch.manual_seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

dataset = HeterophilousGraphDataset(root='data/Amazon-ratings', name='Amazon-ratings')
data = dataset[0]

# Match original preprocessing intent:
# - make graph undirected
# - no self-loops for SAGE
edge_index = to_undirected(data.edge_index, num_nodes=data.num_nodes)

x = data.x.to(device)
y = data.y.to(device)
edge_index = edge_index.to(device)

train_mask = data.train_mask.to(device)  # [num_nodes, num_splits]
val_mask = data.val_mask.to(device)
test_mask = data.test_mask.to(device)

print("num_nodes:", data.num_nodes)
print("num_edges (after to_undirected):", edge_index.size(1))
print("num_features:", dataset.num_features)
print("num_classes:", dataset.num_classes)
print("num_splits:", train_mask.size(1))

In [ ]:
import torch.nn as nn

class ResidualModuleWrapper(nn.Module):
    def __init__(self, module_cls, normalization_cls, dim, **kwargs):
        super().__init__()
        self.normalization = normalization_cls(dim)
        self.module = module_cls(dim=dim, **kwargs)

    def forward(self, x, edge_index):
        x_res = self.normalization(x)
        x_res = self.module(x_res, edge_index)
        return x + x_res


class FeedForwardModule(nn.Module):
    def __init__(self, dim, hidden_dim_multiplier=1.0, dropout=0.2, input_dim_multiplier=1.0):
        super().__init__()
        input_dim = int(dim * input_dim_multiplier)
        hidden_dim = int(dim * hidden_dim_multiplier)

        self.linear_1 = nn.Linear(input_dim, hidden_dim)
        self.dropout_1 = nn.Dropout(dropout)
        self.act = nn.GELU()
        self.linear_2 = nn.Linear(hidden_dim, dim)
        self.dropout_2 = nn.Dropout(dropout)

    def forward(self, x):
        x = self.linear_1(x)
        x = self.dropout_1(x)
        x = self.act(x)
        x = self.linear_2(x)
        x = self.dropout_2(x)
        return x


class SAGEModule(nn.Module):
    # message = mean over incoming neighbors; concat [x, message] and project back to dim
    def __init__(self, dim, hidden_dim_multiplier=1.0, dropout=0.2):
        super().__init__()
        self.ffn = FeedForwardModule(
            dim=dim,
            input_dim_multiplier=2.0,
            hidden_dim_multiplier=hidden_dim_multiplier,
            dropout=dropout
        )

    def forward(self, x, edge_index):
        src, dst = edge_index
        n = x.size(0)

        msg_sum = torch.zeros_like(x)
        msg_sum.index_add_(0, dst, x[src])

        deg = torch.zeros(n, device=x.device, dtype=x.dtype)
        deg.index_add_(0, dst, torch.ones_like(dst, dtype=x.dtype))
        deg = deg.clamp(min=1.0).unsqueeze(-1)

        message = msg_sum / deg
        out = torch.cat([x, message], dim=1)
        out = self.ffn(out)
        return out


class SAGE(nn.Module):
    # Mirrors the repo's Model(..., model_name='SAGE') structure.
    def __init__(self, input_dim, output_dim, num_layers=5, hidden_dim=512, hidden_dim_multiplier=1.0, normalization='LayerNorm', dropout=0.2):
        super().__init__()

        normalization_map = {
            "None": nn.Identity,
            "LayerNorm": nn.LayerNorm,
            "BatchNorm": nn.BatchNorm1d
        }
        normalization_cls = normalization_map[normalization]

        self.input_linear = nn.Linear(input_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.act = nn.GELU()

        self.residual_modules = nn.ModuleList([
            ResidualModuleWrapper(
                module_cls=SAGEModule,
                normalization_cls=normalization_cls,
                dim=hidden_dim,
                hidden_dim_multiplier=hidden_dim_multiplier,
                dropout=dropout
            )
            for _ in range(num_layers)
        ])

        self.output_norm = normalization_cls(hidden_dim)
        self.output_linear = nn.Linear(hidden_dim, output_dim)

    def forward(self, x, edge_index):
        x = self.input_linear(x)
        x = self.dropout(x)
        x = self.act(x)

        for block in self.residual_modules:
            x = block(x, edge_index)

        x = self.output_norm(x)
        x = self.output_linear(x)
        return x

In [ ]:
import time
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import f1_score

def macro_f1_score_torch(y_true, y_pred):
    return f1_score(
        y_true.detach().cpu().numpy(),
        y_pred.detach().cpu().numpy(),
        average='macro',
        zero_division=0
    )

def get_parameter_groups(model):
    # same no-weight-decay logic as original utilities
    no_weight_decay_names = ["bias", "normalization", "norm", "label_embeddings"]
    return [
        {
            "params": [
                p for n, p in model.named_parameters()
                if not any(k in n for k in no_weight_decay_names)
            ]
        },
        {
            "params": [
                p for n, p in model.named_parameters()
                if any(k in n for k in no_weight_decay_names)
            ],
            "weight_decay": 0.0
        }
    ]

def get_lr_scheduler_with_warmup(optimizer, num_steps, num_warmup_steps=None, warmup_proportion=0.0):
    if num_warmup_steps is None:
        num_warmup_steps = int(num_steps * warmup_proportion)

    def lr_lambda(step):
        if step < num_warmup_steps:
            return (step + 1) / (num_warmup_steps + 1)
        return 1.0

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)

def run_one_depth(
    num_layers=5,
    num_steps=1000,
    lr=3e-5,
    weight_decay=0.0,
    dropout=0.2,
    eval_every=10,
    early_stop=True,
    patience=20,       # number of eval checks with no val improvement
    min_delta=1e-4,    # minimum improvement to reset patience
    min_steps=100,     # do not early-stop too early
    hidden_dim=512,
    hidden_dim_multiplier=1.0,
    normalization='LayerNorm'
    ):
    num_splits = train_mask.size(1)

    best_val_scores = []
    best_test_scores = []
    best_steps = []
    best_val_f1_scores = []
    best_test_f1_scores = []

    t0 = time.perf_counter()

    for split_idx in range(num_splits):
        model = SAGE(
            input_dim=dataset.num_features,
            output_dim=dataset.num_classes,
            num_layers=num_layers,
            hidden_dim=hidden_dim,
            hidden_dim_multiplier=hidden_dim_multiplier,
            normalization=normalization,
            dropout=dropout
        ).to(device)

        parameter_groups = get_parameter_groups(model)
        optimizer = torch.optim.AdamW(parameter_groups, lr=lr, weight_decay=weight_decay)
        scheduler = get_lr_scheduler_with_warmup(
            optimizer=optimizer,
            num_steps=num_steps,
            num_warmup_steps=None,
            warmup_proportion=0.0
        )
        criterion = nn.CrossEntropyLoss()

        tr = train_mask[:, split_idx]
        va = val_mask[:, split_idx]
        te = test_mask[:, split_idx]

        best_val = -1.0
        best_test = -1.0
        best_step = 0
        best_val_f1 = 0.0
        best_test_f1 = 0.0
        bad_eval_count = 0

        for step in range(1, num_steps + 1):
            model.train()
            optimizer.zero_grad()
            logits = model(x, edge_index)
            loss = criterion(logits[tr], y[tr])
            loss.backward()
            optimizer.step()
            scheduler.step()

            if step == 1 or step % eval_every == 0:
                model.eval()
                with torch.no_grad():
                    logits = model(x, edge_index)
                    pred = logits.argmax(dim=1)

                    train_acc = (pred[tr] == y[tr]).float().mean().item()
                    val_acc = (pred[va] == y[va]).float().mean().item()
                    test_acc = (pred[te] == y[te]).float().mean().item()

                    train_f1 = macro_f1_score_torch(y[tr], pred[tr])
                    val_f1 = macro_f1_score_torch(y[va], pred[va])
                    test_f1 = macro_f1_score_torch(y[te], pred[te])

                train_err = 1.0 - train_acc
                val_err = 1.0 - val_acc
                test_err = 1.0 - test_acc

                improved = (val_acc - best_val) > min_delta
                if improved:
                    best_val = val_acc
                    best_test = test_acc
                    best_step = step
                    best_val_f1 = val_f1
                    best_test_f1 = test_f1
                    bad_eval_count = 0
                else:
                    bad_eval_count += 1

                print(
                    f"L={num_layers} | split={split_idx+1:02d}/{num_splits} | "
                    f"step={step:04d} | loss={loss.item():.4f} | "
                    f"train_err={train_err:.4f} | val_err={val_err:.4f} | test_err={test_err:.4f} | "
                    f"train_f1={train_f1:.4f} | val_f1={val_f1:.4f} | test_f1={test_f1:.4f}"
                )

                if early_stop and step >= min_steps and bad_eval_count >= patience:
                    print(
                        f"Early stop: no val improvement for {patience} evals "
                        f"(best_val={best_val:.4f} at step {best_step})."
                    )
                    break

        best_val_scores.append(best_val)
        best_test_scores.append(best_test)
        best_steps.append(best_step)
        best_val_f1_scores.append(best_val_f1)
        best_test_f1_scores.append(best_test_f1)

        print(
            f"Best split {split_idx+1:02d}: "
            f"val_acc={best_val:.4f}, test_acc={best_test:.4f}, "
            f"val_f1={best_val_f1:.4f}, test_f1={best_test_f1:.4f}, step={best_step}"
        )

    elapsed = time.perf_counter() - t0

    val_mean = float(np.mean(best_val_scores))
    val_std = float(np.std(best_val_scores, ddof=1))
    test_mean = float(np.mean(best_test_scores))
    test_std = float(np.std(best_test_scores, ddof=1))

    val_f1_mean = float(np.mean(best_val_f1_scores))
    val_f1_std = float(np.std(best_val_f1_scores, ddof=1))
    test_f1_mean = float(np.mean(best_test_f1_scores))
    test_f1_std = float(np.std(best_test_f1_scores, ddof=1))

    return {
        "num_layers": num_layers,
        "val_mean": val_mean,
        "val_std": val_std,
        "test_mean": test_mean,
        "test_std": test_std,
        "val_f1_mean": val_f1_mean,
        "val_f1_std": val_f1_std,
        "test_f1_mean": test_f1_mean,
        "test_f1_std": test_f1_std,
        "best_steps": best_steps,
        "elapsed_sec": elapsed,
    }

## Hyperparameter Tuning
In this section, we use Optuna to search for good SAGE hyperparameters automatically.
The search space is designed to include the known reference setup, so that configuration can be selected during optimization.

In [ ]:
# Optional if Optuna is missing:
# %pip install -q optuna

import optuna

REFERENCE_HP = {
    "num_layers": 2,
    "num_steps": 1000,
    "lr": 3e-5,
    "weight_decay": 0.0,
    "dropout": 0.2,
    "eval_every": 10,
    "patience": 20,
    "min_delta": 1e-4,
    "min_steps": 100,
}

SEARCH_SPACE = {
    "num_layers": [1, 2, 3, 4, 5],
    "num_steps": [600, 800, 1000, 1200],
    "lr": [1e-5, 2e-5, 3e-5, 5e-5, 1e-4],
    "weight_decay": [0.0, 1e-6, 1e-5, 1e-4],
    "dropout": [0.1, 0.2, 0.3, 0.4],
    "eval_every": [5, 10, 20],
    "patience": [10, 15, 20, 25, 30],
    "min_delta": [0.0, 1e-5, 1e-4, 5e-4, 1e-3],
    "min_steps": [50, 100, 150, 200],
}

run_one_depth_fn = globals().get("run_one_depth")
if run_one_depth_fn is None:
    print("run_one_depth is not defined yet. Run the training/model cells first, then rerun this cell.")
else:
    def objective(trial):
        params = {
            key: trial.suggest_categorical(key, choices)
            for key, choices in SEARCH_SPACE.items()
        }

        result = run_one_depth_fn(
            num_layers=params["num_layers"],
            num_steps=params["num_steps"],
            lr=params["lr"],
            weight_decay=params["weight_decay"],
            dropout=params["dropout"],
            eval_every=params["eval_every"],
            early_stop=True,
            patience=params["patience"],
            min_delta=params["min_delta"],
            min_steps=params["min_steps"],
        )

        trial.set_user_attr("val_f1_mean", result["val_f1_mean"])
        trial.set_user_attr("test_acc_mean", result["test_mean"])
        trial.set_user_attr("test_f1_mean", result["test_f1_mean"])
        return result["val_mean"]

    sampler = optuna.samplers.TPESampler(seed=42)
    study = optuna.create_study(
        direction="maximize",
        sampler=sampler,
        study_name="sage_hyperparam_tuning"
    )

    n_trials = 10
    study.optimize(objective, n_trials=n_trials)

    best_trial = study.best_trial
    print(f"Best validation accuracy: {best_trial.value:.4f}")
    print(f"Best validation macro-F1: {best_trial.user_attrs['val_f1_mean']:.4f}")
    print(f"Best test accuracy: {best_trial.user_attrs['test_acc_mean']:.4f}")
    print(f"Best test macro-F1: {best_trial.user_attrs['test_f1_mean']:.4f}")
    print("Best hyperparameters:")
    for key, value in best_trial.params.items():
        print(f"  {key}: {value}")

In [ ]:
# Final paper-style sweep: depths 1..5, 1000 steps, full split averaging.
results = []
for L in [1, 2, 3, 4, 5]:
    print("\n" + "=" * 100)
    print(f"Running SAGE depth L={L}")

    r = run_one_depth(
        num_layers=L,
        num_steps=1000,
        lr=3e-5,
        weight_decay=0.0,
        dropout=0.2,
        eval_every=10,
        early_stop=True,
        patience=20,
        min_delta=1e-4,
        min_steps=100
    )

    results.append(r)
    print(
        f"L={L} -> val_acc={r['val_mean']*100:.2f} +/- {r['val_std']*100:.2f}, "
        f"test_acc={r['test_mean']*100:.2f} +/- {r['test_std']*100:.2f}, "
        f"val_f1={r['val_f1_mean']*100:.2f} +/- {r['val_f1_std']*100:.2f}, "
        f"test_f1={r['test_f1_mean']*100:.2f} +/- {r['test_f1_std']*100:.2f}, "
        f"time={r['elapsed_sec']:.1f}s"
    )

best = max(results, key=lambda t: t["val_mean"])
print("\n" + "#" * 100)
print(f"Best depth by val mean: L={best['num_layers']}")
print(f"Final test accuracy: {best['test_mean']*100:.2f} +/- {best['test_std']*100:.2f}")
print(f"Final test macro-F1: {best['test_f1_mean']*100:.2f} +/- {best['test_f1_std']*100:.2f}")
print("Reference from paper table for SAGE on amazon-ratings can vary by implementation/runtime.")

## Ablation and Error Analysis on Graph Topology
After training, this section analyzes where the model performs well or poorly.
We break down results by node degree and local neighborhood label agreement to interpret failure modes.

In [ ]:
# --- 1. EXTRACT PREDICTIONS FROM BEST CONFIGURATION (L=2) ---
import copy
import torch
import torch.nn as nn
from sklearn.metrics import f1_score

L_best = 2
split_idx = 0 # Using the first split for detailed error analysis

# Isolate the masks for split 0
tr = data.train_mask[:, split_idx]
va = data.val_mask[:, split_idx]
te = data.test_mask[:, split_idx]

# Initialize model using your exact parameters
model_eval = SAGE(
    input_dim=dataset.num_features,
    output_dim=dataset.num_classes,
    num_layers=L_best,
    hidden_dim=512,
    hidden_dim_multiplier=1.0,
    normalization='LayerNorm',
    dropout=0.2,
).to(device)

optimizer = torch.optim.AdamW(get_parameter_groups(model_eval), lr=3e-5, weight_decay=0.0)
criterion = nn.CrossEntropyLoss()

best_val = -1.0
best_model_state = None

print(f"Training L={L_best} on Split {split_idx} to extract predictions...")

# Train loop based on run_one_depth logic
for step in range(1, 1001):
    model_eval.train()
    optimizer.zero_grad()
    logits = model_eval(x, edge_index)
    loss = criterion(logits[tr], y[tr])
    loss.backward()
    optimizer.step()

    # Evaluation and checkpointing
    if step % 10 == 0:
        model_eval.eval()
        with torch.no_grad():
            logits = model_eval(x, edge_index)
            pred = logits.argmax(dim=1)
            val_acc = (pred[va] == y[va]).float().mean().item()

            if val_acc > best_val:
                best_val = val_acc
                best_model_state = copy.deepcopy(model_eval.state_dict())

# Load the best weights based on validation accuracy
model_eval.load_state_dict(best_model_state)
model_eval.eval()

with torch.no_grad():
    logits = model_eval(x, edge_index)
    final_pred = logits.argmax(dim=1)

    # Create a boolean mask of correct predictions on the test set
    correct_mask_tensor = (final_pred[te] == y[te])
    base_test_acc = correct_mask_tensor.float().mean().item()
    base_test_f1 = f1_score(
        y[te].detach().cpu().numpy(),
        final_pred[te].detach().cpu().numpy(),
        average='macro',
        zero_division=0
    )

print(f"Base Model Test Acc (Split 0): {base_test_acc*100:.2f}%")
print(f"Base Model Test Macro-F1 (Split 0): {base_test_f1*100:.2f}%")

In [ ]:
# --- 2. ERROR ANALYSIS: NODE DEGREE ---
import matplotlib.pyplot as plt
import numpy as np
from torch_geometric.utils import degree

# 1. Calculate degree for all nodes
node_degrees = degree(edge_index[0], num_nodes=x.size(0)).cpu().numpy()

# 2. Isolate test nodes and their correctness
test_indices = te.nonzero(as_tuple=False).view(-1).cpu()
test_degrees = node_degrees[test_indices.numpy()]
correct_preds = correct_mask_tensor.cpu().numpy()

# 3. Bin the degrees
bins = [0, 2, 5, 10, 20, 50, 100, np.inf]
labels = ['0-2', '3-5', '6-10', '11-20', '21-50', '51-100', '100+']
digitized = np.digitize(test_degrees, bins)

acc_by_bin = []
for i in range(1, len(bins)):
    bin_filter = (digitized == i)
    if bin_filter.sum() > 0:
        acc = correct_preds[bin_filter].mean()
    else:
        acc = 0.0
    acc_by_bin.append(acc)

# 4. Plot
plt.figure(figsize=(10, 5))
plt.bar(labels, acc_by_bin, color='#4C72B0', edgecolor='black')
plt.xlabel('Node Degree (Outliers → Hubs)')
plt.ylabel('Test Accuracy')
plt.title('SAGE Performance by Node Degree (Amazon-Ratings)')
plt.ylim(0, 1.1)

for i, acc in enumerate(acc_by_bin):
    if acc > 0:
        plt.text(i, acc + 0.02, f"{acc:.2f}", ha='center', fontsize=9)
plt.show()

In [ ]:
# --- 3. ERROR ANALYSIS: LOCAL HETEROPHILY ---
from collections import defaultdict

edge_index_cpu = edge_index.cpu()
y_cpu = y.cpu()
num_nodes = x.size(0)

# 1. Map neighbors for each node
neighbors = defaultdict(list)
for i in range(edge_index_cpu.size(1)):
    u = edge_index_cpu[0, i].item()
    v = edge_index_cpu[1, i].item()
    neighbors[u].append(v)

# 2. Calculate localized homophily ratio per node
same_class_ratios = np.zeros(num_nodes)
for node in range(num_nodes):
    nbs = neighbors[node]
    if len(nbs) > 0:
        same_class_count = sum([1 for nb in nbs if y_cpu[nb] == y_cpu[node]])
        same_class_ratios[node] = same_class_count / len(nbs)
    else:
        same_class_ratios[node] = -1 # Flag isolated nodes

# 3. Filter for test set and bin ratios
test_ratios = same_class_ratios[test_indices.numpy()]
ratio_bins = np.linspace(0, 1, 6)
ratio_labels = ['0.0-0.2', '0.2-0.4', '0.4-0.6', '0.6-0.8', '0.8-1.0']
digitized_ratios = np.digitize(test_ratios, ratio_bins[1:], right=True)

acc_by_ratio = []
for i in range(len(ratio_labels)):
    bin_filter = (digitized_ratios == i) & (test_ratios >= 0)
    if bin_filter.sum() > 0:
        acc = correct_preds[bin_filter].mean()
    else:
        acc = 0.0
    acc_by_ratio.append(acc)

# 4. Plot
plt.figure(figsize=(10, 5))
plt.bar(ratio_labels, acc_by_ratio, color='#C44E52', edgecolor='black')
plt.xlabel('Proportion of Same-Class Neighbors (Heterophilic → Homophilic)')
plt.ylabel('Test Accuracy')
plt.title('The Heterophily Challenge: SAGE Accuracy vs. Neighborhood Match')
plt.ylim(0, 1.1)

for i, acc in enumerate(acc_by_ratio):
    if acc > 0:
        plt.text(i, acc + 0.02, f"{acc:.2f}", ha='center', fontsize=9)
plt.show()